In [2]:
!pip install pandas numpy matplotlib seaborn plotly scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from google.colab import files

df = pd.read_csv("ecommerce_business_dataset.csv")

df['order_date'] = pd.to_datetime(df['order_date'])

total_revenue = df['revenue'].sum()
total_orders = df['order_id'].nunique()
total_customers = df['customer_id'].nunique()
average_order_value = total_revenue / total_orders

print("Total Revenue:", total_revenue)
print("Total Orders:", total_orders)
print("Total Customers:", total_customers)
print("Average Order Value:", average_order_value)

df['month'] = df['order_date'].dt.to_period('M')
monthly_revenue = df.groupby('month')['revenue'].sum().reset_index()
monthly_revenue['month'] = monthly_revenue['month'].astype(str)

fig = px.line(monthly_revenue, x='month', y='revenue', title="Monthly Revenue Trend")
fig.show()

country_revenue = df.groupby('country')['revenue'].sum().sort_values(ascending=False)

country_revenue.plot(kind='bar')
plt.title("Revenue by Country")
plt.show()

df['OrderMonth'] = df['order_date'].dt.to_period('M')
df['CohortMonth'] = df.groupby('customer_id')['OrderMonth'].transform('min')

cohort_data = df.groupby(['CohortMonth','OrderMonth'])['customer_id'].nunique().reset_index()

cohort_table = cohort_data.pivot(index='CohortMonth', columns='OrderMonth', values='customer_id')

plt.figure(figsize=(10,6))
sns.heatmap(cohort_table, cmap="Blues")
plt.title("Cohort Analysis")
plt.show()

snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)

rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'revenue': 'sum'
})

rfm.columns = ['Recency','Frequency','Monetary']

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Segment'] = kmeans.fit_predict(rfm_scaled)

fig = px.scatter(rfm, x='Frequency', y='Monetary', color='Segment', title="Customer Segmentation")
fig.show()

category_revenue = df.groupby('category')['revenue'].sum().reset_index()

fig = px.pie(category_revenue, values='revenue', names='category', title="Revenue by Category")
fig.show()

payment_revenue = df.groupby('payment_method')['revenue'].sum().reset_index()

fig = px.bar(payment_revenue, x='payment_method', y='revenue', title="Revenue by Payment Method")
fig.show()

funnel = pd.DataFrame({
    'Stage': ['Visitors','Product View','Add To Cart','Purchase'],
    'Users': [10000,7000,3500,1500]
})

fig = px.funnel(funnel, x='Users', y='Stage', title="Customer Funnel")
fig.show()

rfm.to_csv("customer_segments.csv")
monthly_revenue.to_csv("monthly_revenue.csv")

files.download("customer_segments.csv")
files.download("monthly_revenue.csv")
